# 应用负载预测模型训练

## 目标
- 使用多种算法训练负载预测模型
- 评估和比较模型性能
- 分析特征重要性

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 添加项目路径
sys.path.append(os.path.abspath('..'))

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 从本地模块导入
from src.preprocessing.data_loader import DataLoader
from src.models.prediction_model import LoadPredictionModel
from src.visualization.plotter import LoadVisualizer

print("导入成功！")

## 1. 加载处理后的数据

In [ ]:
# 加载数据
loader = DataLoader(data_dir='../data/processed')
df = loader.load_csv('processed_load_data.csv', parse_dates=['timestamp'])

print(f"数据形状：{df.shape}")
print(f"列名：{df.columns.tolist()}")
df.head()

## 2. 准备训练数据

In [ ]:
# 定义目标变量
target_column = 'cpu_usage'

# 初始化模型
model_handler = LoadPredictionModel(model_type='xgboost')

# 准备数据
X_train, X_test, y_train, y_test = model_handler.prepare_data(
    df, 
    target_column, 
    test_size=0.2
)

print(f"训练集大小：{len(X_train)}")
print(f"测试集大小：{len(X_test)}")

## 3. 训练 XGBoost 模型

In [ ]:
# 训练 XGBoost
print("开始训练 XGBoost 模型...")
model_handler.train_xgboost(X_train, y_train)

# 评估模型
metrics_xgb = model_handler.evaluate(X_test, y_test)
print(f"\nXGBoost 模型指标：{metrics_xgb}")

In [ ]:
# 获取特征重要性
feature_names = X_train.columns.tolist()
importance_df = model_handler.get_feature_importance(feature_names, top_n=15)

# 可视化特征重要性
visualizer = LoadVisualizer()
visualizer.plot_feature_importance(importance_df, top_n=15, title='XGBoost 特征重要性')

In [ ]:
# 预测值 vs 实际值对比
y_pred = model_handler.model.predict(X_test)
visualizer.plot_prediction_vs_actual(y_test, y_pred, 'XGBoost 预测结果')

## 4. 训练随机森林模型（对比）

In [ ]:
# 训练随机森林
print("开始训练随机森林模型...")
model_handler.train_random_forest(X_train, y_train)

# 评估模型
metrics_rf = model_handler.evaluate(X_test, y_test)
print(f"\n随机森林模型指标：{metrics_rf}")

## 5. 模型比较

In [ ]:
# 比较不同模型的性能
comparison_df = pd.DataFrame({
    'XGBoost': metrics_xgb,
    'Random Forest': metrics_rf
})

print("\n模型性能对比:")
print(comparison_df)

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_names = ['RMSE', 'MAE', 'R2']
for i, metric in enumerate(metrics_names):
    axes[i].bar(['XGBoost', 'Random Forest'], 
                [comparison_df.loc[metric, 'XGBoost'], 
                 comparison_df.loc[metric, 'Random Forest']])
    axes[i].set_title(f'{metric} 对比', fontsize=14)
    axes[i].set_ylabel(metric)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('模型性能对比', fontsize=16)
plt.tight_layout()
plt.show()

## 6. 保存模型（可选）

In [ ]:
# TODO: 保存最佳模型
# import joblib
# joblib.dump(model_handler.model, '../models/best_model.pkl')
# print("模型已保存")

## 总结

- ✅ 完成了 XGBoost 和随机森林模型的训练
- ✅ 评估了模型性能
- ✅ 分析了特征重要性
- 📊 选择了表现最佳的模型用于生产环境